# xlingual-drift: Colab runner

GPU-only steps run here: serving open-weight models with vLLM, and later the SFT ablation.
Everything else (environment, verifier, stats) is developed locally.

Workflow: clone from GitHub each session, write raw logs to Drive (they survive
disconnects), commit condensed results back to GitHub before the session ends.
Runs are resumable, so after a disconnect just rerun from the top.

In [ ]:
# 1. Confirm the GPU. Qwen 3.6 27B in BF16 needs the 80GB A100.
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
vram_mib = int(out.split(",")[-1])
assert vram_mib > 70000, "Need the 80GB A100 for BF16. Change the runtime type and reconnect."

In [ ]:
# 2. Clone or update the repo.
import os
REPO_URL = "https://github.com/khalerishabh/xlingual-drift.git"
if not os.path.exists("/content/xlingual-drift"):
    !git clone {REPO_URL} /content/xlingual-drift
else:
    !cd /content/xlingual-drift && git pull
%cd /content/xlingual-drift
!git log --oneline -1

In [ ]:
# 3. Install. vLLM is Colab-only, so it is not in requirements.txt.
!pip install -q -r requirements.txt
!pip install -q "vllm>=0.17.0"

In [ ]:
# 4. Raw logs go to Drive so a disconnect never loses finished episodes.
from google.colab import drive
drive.mount("/content/drive")
RUNS_DIR = "/content/drive/MyDrive/xlingual-drift-runs"
os.makedirs(RUNS_DIR, exist_ok=True)

## Start the model server

Qwen 3.6 27B uses the `qwen3_coder` tool-call parser and the `qwen3` reasoning parser
(vLLM recipe). The first start downloads roughly 55GB of weights, which takes a while.
Server output goes to `/content/vllm.log`.

For Gemma 4 31B instead: `--tool-call-parser gemma4 --reasoning-parser gemma4` plus the
Gemma 4 tool chat template from the vLLM repo (see the vLLM Gemma 4 recipe).

In [ ]:
# 5. Start vLLM in the background and wait until it answers.
import time, urllib.request

MODEL = "Qwen/Qwen3.6-27B"
cmd = ["vllm", "serve", MODEL,
       "--port", "8000",
       "--max-model-len", "32768",
       "--gpu-memory-utilization", "0.90",
       "--enable-auto-tool-choice",
       "--tool-call-parser", "qwen3_coder",
       "--reasoning-parser", "qwen3"]
server = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

for i in range(240):
    try:
        if urllib.request.urlopen("http://localhost:8000/health", timeout=5).status == 200:
            print(f"server up after ~{i * 10}s")
            break
    except Exception:
        pass
    if server.poll() is not None:
        raise RuntimeError("vLLM exited; see the tail of /content/vllm.log below")
    time.sleep(10)
else:
    raise TimeoutError("server not up after 40 min")

In [ ]:
# If the server failed to start, the reason is here.
!tail -n 30 /content/vllm.log

## Checks before spending GPU time

In [ ]:
# 6. Harness sanity: tests and the mock smoke test (no model involved).
!python -m pytest -q
!python -m eval.run_grid --config configs/smoke_test.yaml --out /content/smoke_test.jsonl | tail -2

## Capability gate

120 episodes: 4 horizons x 5 languages x thinking on/off x 3 seeds.
Safe to rerun after a disconnect: `--resume` skips finished episodes.

In [ ]:
# 7. Run the gate.
LOG = f"{RUNS_DIR}/capability_gate_qwen36_27b.jsonl"
!python -m eval.run_grid --config configs/capability_gate.yaml --resume --out {LOG}

In [ ]:
# 8. Clean-episode success by cell x language x horizon. The gate reads the h2 column.
import json, collections
from eval.metrics import success_table

latest = {}
for line in open(LOG, encoding="utf-8"):
    rec = json.loads(line)
    if not rec.get("infra_error"):
        latest[rec["episode_key"]] = rec
records = list(latest.values())
print(f"{len(records)} scored episodes")

table = success_table(records)
langs = ["en", "zh", "hi", "ta", "hinglish"]
for cell in sorted({k[0] for k in table}):
    print(f"\n{cell}")
    print("  lang      " + "  ".join(f"h{h}  " for h in (2, 4, 6, 8)))
    for lang in langs:
        row = [table.get((cell, lang, h)) for h in (2, 4, 6, 8)]
        print(f"  {lang:<9} " + "  ".join(f"{r[0]:.2f}" if r else " -- " for r in row))

tokens = collections.defaultdict(list)
for r in records:
    tokens[r["cell"]].append(r["transcript"]["usage"]["completion_tokens"])
for cell, v in tokens.items():
    print(f"{cell}: mean completion tokens per episode = {sum(v) / len(v):.0f}")

In [ ]:
# 9. What went wrong in failed episodes: booked flight vs decoys, first error, steps.
for r in sorted(records, key=lambda r: (r["cell"], r["language"], r["horizon"], r["seed"])):
    if not r["success"]:
        err = next((s["error"]["message"] for s in r["steps"] if "error" in s), "")
        print(f'{r["cell"]:<20} {r["language"]:<9} h{r["horizon"]} s{r["seed"]} booked={r["booked_flight"]} '
              f'steps={r["num_steps_taken"]} limit={r["hit_step_limit"]} first_err={err[:70]}')

## Save the condensed result to GitHub

Raw logs stay on Drive. This commits only the summary table. Needs a GitHub
personal access token (repo scope) stored in Colab Secrets as `GITHUB_TOKEN`.

In [ ]:
# 10. Commit the summary.
from google.colab import userdata
os.makedirs("results", exist_ok=True)
summary = {f"{c}|{l}|h{h}": {"success": s, "n": n} for (c, l, h), (s, n) in table.items()}
json.dump(summary, open("results/capability_gate_qwen36_27b.json", "w"), indent=2)

token = userdata.get("GITHUB_TOKEN")
!git config user.name "khalerishabh"
!git config user.email "khalerishabh998@gmail.com"
!git add results/capability_gate_qwen36_27b.json
!git commit -m "Capability gate: Qwen 3.6 27B, travel_017, 5 languages" || echo "nothing to commit"
!git push https://{token}@github.com/khalerishabh/xlingual-drift.git HEAD:main